# Notebook 02a: Phân tích Chuyên sâu Câu hỏi Q1 (VeDDRA & Time Series)
**Đề tài:** Phân tích Sự cố Bất lợi Thuốc Thú y (OpenFDA Animal & Veterinary Adverse Events)

**Môn học:** CSC17104 – Lập trình cho Khoa học Dữ liệu  
**Thành viên thực hiện:** Đinh Công Khang (M1) — Nhánh Git: `feature/khang`  
**Phạm vi sở hữu:** **Câu hỏi Q1** (Non-ML Question)

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập đường dẫn gốc dự án
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_processed
from src.preprocessing import log_row_count_change
from src.visualization import set_project_plot_style
set_project_plot_style()

In [ ]:
# Tải dữ liệu đã qua tiền xử lý chuẩn hóa từ data/processed/
data = load_processed(PROJECT_ROOT / 'data' / 'processed')
reports = data['reports']
reactions = data['reactions']
drugs = data['drugs']
outcomes = data['outcomes']

print(f"[Data Loaded] Reports: {reports.shape}, Reactions: {reactions.shape}, Drugs: {drugs.shape}, Outcomes: {outcomes.shape}")

---
## Câu hỏi Q1: Phản ứng VeDDRA hàng đầu & Xu hướng biến động theo quý

### Phát biểu câu hỏi & Động lực (Question & Motivation)
- **Phát biểu câu hỏi:**  
  > **"Những phản ứng bất lợi (VeDDRA terms) nào chiếm tỷ trọng cao nhất ở Chó, Mèo và các loài gia súc/vật nuôi khác trong năm 2025, và xu hướng theo quý (kèm dấu hiệu mùa vụ trong phạm vi 1 năm) của các phản ứng hàng đầu này biến động như thế nào cho đến nửa đầu năm 2026 (Q1)?"**
- **Người quan tâm & Quyết định hỗ trợ (Target Users & Decisions):**
  - **Bác sĩ thú y lâm sàng:** Cần nắm bắt mô hình triệu chứng phổ biến nhất theo từng loài cụ thể để chẩn đoán phân biệt và nhận diện sớm các phản ứng bất lợi sau điều trị theo từng mùa trong năm.
  - **Cơ quan quản lý (FDA CVM) & Nhà sản xuất thuốc:** Cần giám sát tín hiệu an toàn thời gian thực để phát hiện các đợt bùng phát bất thường (outbreaks) hoặc sự gia tăng đột biến của phản ứng phụ theo quý.
- **Tiêu chí độ sâu áp dụng (Mục 3.3):**
  - `D1` (Combining data): Nối bảng báo cáo cha `reports` với bảng triệu chứng lâm sàng con `reactions` theo khóa `unique_aer_id_number` (kiểm tra số dòng trước/sau và tỷ lệ khớp).
  - `D2` (Reshaping & time): Resample dữ liệu theo quý (2025 Q1 — 2026 Q1), xoay bảng (pivot table) tần suất phản ứng theo thời gian và tính tỷ lệ biến động qua từng kỳ.
  - `D5` (Quantifying uncertainty): Ước lượng khoảng tin cậy 95% Bootstrap cho tỷ lệ phản ứng theo quý và thực hiện kiểm định $\chi^2$ về tính đồng nhất phân bố mùa vụ.

### 1. Tiền xử lý dữ liệu cho Q1 (Preprocessing & Row Count Tracking)

*(Quy tắc: Mô tả rõ ràng từng bước bằng Markdown trước khi viết code. Sau mỗi bước lọc/gộp/loại bỏ, ghi nhận số dòng bằng `log_row_count_change`)*:
1. **Bước 1.1:** Lọc các báo cáo có loài xác định (`species` thuộc nhóm Chó, Mèo, hoặc nhóm loài khác được gom cụm).
2. **Bước 1.2:** Kết hợp `reports` với bảng `reactions` qua `unique_aer_id_number`, loại bỏ các bản ghi không có thông tin `veddra_term_name` hợp lệ.
3. **Bước 1.3:** Tạo biến định danh quý `quarter` chuẩn hóa (`2025Q1`, `2025Q2`, `2025Q3`, `2025Q4`, `2026Q1`).

In [ ]:
# TODO [Khang - M1]: Triển khai các bước tiền xử lý cho Q1 kèm log_row_count_change
initial_count = len(reports)
# Ví dụ: filtered_df = ...
# log_row_count_change('Q1: Lọc loài hợp lệ', initial_count, len(filtered_df), 'Loại bỏ loài thiếu hoặc không xác định')

### 2. Phương pháp phân tích (Analysis Method)

*(Giải thích phương pháp được chọn và lý do phương pháp này phù hợp để giải quyết câu hỏi)*:
- **Phương pháp 1:** Xếp hạng Top 10 phản ứng VeDDRA có tần suất xuất hiện cao nhất, phân tầng độc lập cho Chó, Mèo và các loài khác trong năm 2025.
- **Phương pháp 2:** Tính toán tỷ lệ phần trăm của từng phản ứng trong tổng số báo cáo của từng quý từ 2025 Q1 đến 2026 Q1.
- **Phương pháp 3:** Tính khoảng tin cậy 95% Bootstrap cho tỷ lệ phản ứng hàng đầu theo từng quý và kiểm định $\chi^2$ kiểm tra xem xu hướng có thay đổi có ý nghĩa thống kê giữa các quý không.

In [ ]:
# TODO [Khang - M1]: Triển khai phân tích dữ liệu Q1 (Xếp hạng VeDDRA, Resampling theo quý, Bootstrap CI)

### 3. Kết quả & Trực quan hóa (Results & Visualizations)

*(Yêu cầu đề tài: Tối thiểu 2 biểu đồ chuẩn mực có đầy đủ tiêu đề, nhãn trục, đơn vị và chú thích)*:
- **Biểu đồ 1:** Barplot phân bố Top 10 thuật ngữ VeDDRA phổ biến nhất theo từng loài (Chó vs Mèo vs Khác).
- **Biểu đồ 2:** Lineplot biến thiên chuỗi thời gian 5 quý (2025 Q1 — 2026 Q1) của các phản ứng chủ đạo kèm dải tin cậy 95% (shaded confidence intervals).

In [ ]:
# TODO [Khang - M1]: Vẽ tối thiểu 2 biểu đồ chuẩn mực cho Q1

### 4. Trả lời Câu hỏi & Thảo luận (Answer, Practical Meaning & Limitations)

- **Câu trả lời tường minh (Explicit Answer with specific numbers):**  
  *(Nêu rõ các con số thống kê cụ thể: Tên phản ứng VeDDRA phổ biến nhất, tỷ lệ % chiếm dụng, độ tăng giảm % theo quý, giá trị p-value của kiểm định thống kê)*.
- **Ý nghĩa thực tiễn & Bất ngờ (Practical Meaning & Surprises):**  
  *(Phát hiện có gì bất ngờ về triệu chứng giữa chó và mèo? Có dấu hiệu mùa vụ rõ rệt ở phản ứng nào không — ví dụ bệnh ngoài da hoặc sốc thuốc mùa nóng?)*
- **Hạn chế của phân tích (Limitations):**  
  *(Ảnh hưởng của cơ chế tự nguyện báo cáo, sự sai khác trong thói quen ghi nhận thuật ngữ VeDDRA của các phòng khám thú y)*.